In [ ]:
# ==================================================
# BLOCK 1 & 2: IMPORT VÀ CONFIG
# ==================================================
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
MODEL_NAME = "ResNet50" # Thay đổi tên model tại đây trong danh sách 15 SOTA

# ==================================================
# BLOCK 3: DATA PREPARATION (2 KỊCH BẢN)
# ==================================================
# 3.1 Không Augmentation
datagen_no_aug = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

# 3.2 Có 4 thuật toán Augmentation (Ví dụ: Xoay, Lật, Phóng to, Chỉnh sáng)
datagen_aug = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,     # Thuật toán 1
    horizontal_flip=True,  # Thuật toán 2
    zoom_range=0.2,        # Thuật toán 3
    brightness_range=[0.8, 1.2] # Thuật toán 4
)

# ... (Khai báo train_gen_no_aug, train_gen_aug, val_gen, test_gen) ...

# ==================================================
# BLOCK 4: HÀM BUILD MODEL & LẤY PARAMETERS
# ==================================================
def build_model(model_name, num_classes):
    if model_name == "ResNet50":
        base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    elif model_name == "MobileNetV2":
        base_model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    # AI sẽ thêm các if/elif cho 15 model SOTA tại đây
    
    x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
    output = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.models.Model(inputs=base_model.input, outputs=output)
    
    total_params = model.count_params()
    print(f"[{model_name}] Total Parameters: {total_params:,}")
    return model, total_params

# ==================================================
# BLOCK 5: HÀM ĐÁNH GIÁ & LƯU HÌNH ẢNH (CHÚ Ý CÚ PHÁP LƯU ẢNH)
# ==================================================
def evaluate_and_save_artifacts(model, test_gen, history, scenario_name):
    # 1. Lưu Learning Curve
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Val')
    plt.title(f'{MODEL_NAME} Accuracy ({scenario_name})')
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{MODEL_NAME} Loss ({scenario_name})')
    plt.savefig(f'{MODEL_NAME}_{scenario_name}_Learning_Curve.png', dpi=300) # LƯU ẢNH
    plt.show()

    # 2. Lưu Confusion Matrix
    y_pred = np.argmax(model.predict(test_gen), axis=1)
    y_true = test_gen.classes
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(test_gen.class_indices.keys()))
    fig, ax = plt.subplots(figsize=(10, 8))
    disp.plot(cmap="Blues", ax=ax, xticks_rotation=45)
    plt.title(f'{MODEL_NAME} Confusion Matrix ({scenario_name})')
    plt.tight_layout()
    plt.savefig(f'{MODEL_NAME}_{scenario_name}_Confusion_Matrix.png', dpi=300) # LƯU ẢNH
    plt.show()

    # 3. Tính toán và in Cột: Accuracy, F1, Recall
    from sklearn.metrics import accuracy_score, f1_score, recall_score
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    
    print(f"--- KẾT QUẢ CHO BẢNG GOOGLE SHEET: {scenario_name} ---")
    print(f"Accuracy: {acc:.4f} | F1-Score: {f1:.4f} | Recall: {recall:.4f}")

# ==================================================
# BLOCK 6: THỰC THI CHẠY 2 KỊCH BẢN
# ==================================================
# CHẠY KỊCH BẢN 1: KHÔNG AUGMENTATION
print("=== TRAINING SCENARIO 1: NO AUGMENTATION ===")
model_no_aug, params = build_model(MODEL_NAME, test_gen.num_classes)
# ... (compile & fit với train_gen_no_aug) ...
# evaluate_and_save_artifacts(model_no_aug, test_gen, history_1, "No_Aug")

# CHẠY KỊCH BẢN 2: CÓ AUGMENTATION
print("=== TRAINING SCENARIO 2: WITH AUGMENTATION ===")
model_aug, _ = build_model(MODEL_NAME, test_gen.num_classes) # Build lại model mới
# ... (compile & fit với train_gen_aug) ...
# evaluate_and_save_artifacts(model_aug, test_gen, history_2, "With_Aug")